# Software comparison: fastCDS vs every other protein-to-genome tool

This notebook lines fastCDS up **head to head** against the other tools for
mapping protein/domain coordinates to the genome. For **coordinate agreement**
(Table S1) it compares against the two Bioconductor `proteinToGenome` routes -
**ensembldb** (the naive `EnsDb` call) and **GenomicFeatures**
(`proteinToGenome,GRangesList` on a precomputed CDS GRangesList, in-memory) - plus
TransVar and the Ensembl REST API. For **speed and memory** (Table S2) it compares
fastCDS against ensembldb, GenomicFeatures, geneplot, and the Ensembl REST API.
Every tool is run on the **same human Ensembl-86 set**; the heavy ones (ensembldb,
REST, TransVar, and geneplot on the human genome) are measured offline via the
scripts in `benchmarks/` rather than inline, since each takes minutes to hours.

See the [Speed vs other tools](https://github.com/SotoLF/fastCDS/wiki/Performance-and-Benchmarking#speed-vs-other-tools)
wiki section for the comparator-specific notes (why each tool, what each tool's
denominator means, the TransVar envelope-only caveat).

## Setup

In [ ]:
# Force the inline backend — under `jupyter nbconvert --execute` the
# default sometimes lands on Agg, which prints `<Figure …>` instead of
# the actual PNG. The magic call forces module://matplotlib_inline.backend_inline.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass  # Not in IPython (e.g. plain python REPL); ignore.
import matplotlib as mpl
import matplotlib.pyplot as plt

# Every figure saved as PNG also gets a vector PDF (same bbox/dimensions) under
# reproduce_paper/figures/ (right next to these notebooks), so the
# paper figures are available as scalable PDFs without changing any plotting code.
from pathlib import Path as _Path
import os as _os
FIGDIR = (_Path(_os.environ.get("FASTCDS_REPO") or (_Path.home() / "Desktop" / "fastCDS"))
          / "reproduce_paper" / "figures")
FIGDIR.mkdir(parents=True, exist_ok=True)
if not getattr(mpl.figure.Figure.savefig, "_writes_pdf", False):  # guard: don't double-wrap
    _orig_savefig = mpl.figure.Figure.savefig
    def _savefig_both(self, fname, *a, **k):
        _orig_savefig(self, fname, *a, **k)
        s = str(fname)
        if s.lower().endswith(".png"):
            kk = dict(k); kk.pop("dpi", None)              # vector PDF: drop raster dpi
            _orig_savefig(self, str(FIGDIR / (_os.path.splitext(_os.path.basename(s))[0] + ".pdf")), *a, **kk)
    _savefig_both._writes_pdf = True
    mpl.figure.Figure.savefig = _savefig_both

# Paper-ready figure defaults. Tweaks vs matplotlib's stock style:
#   - Larger fonts (10pt body, 11pt axis labels, 12pt title).
#   - Thinner spines + only-left/-bottom by default (less chartjunk).
#   - Subtle horizontal grid; no vertical grid.
#   - tab10 palette but used sparingly — we override per-plot.
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'semibold',
    'axes.titlepad': 10,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.8,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.color': '#e5e7eb',
    'grid.linewidth': 0.8,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'legend.frameon': False,
    'legend.fontsize': 10,
    'lines.linewidth': 2.0,
})

# Colorblind-safe palette (Wong 2011, also used in seaborn's 'colorblind').
COLORS = {
    'fastCDS':   '#0072B2',  # blue
    'ensembldb':   '#009E73',  # bluish green
    'transvar':    '#E69F00',  # orange
    'rest':        '#CC79A7',  # reddish-purple
    'good':        '#009E73',
    'bad':         '#D55E00',  # vermilion (works for colorblind)
    'neutral':     '#56B4E9',
    'highlight':   '#F0E442',
}

import os, sys, re, subprocess
from shutil import which
import pandas as pd
from pathlib import Path

# Set FASTCDS_REPO / FASTCDS_DATA to override when running outside a checkout.
start = Path.cwd()
_repo_env = os.environ.get("FASTCDS_REPO")
REPO = Path(_repo_env) if _repo_env else next(
    (p for p in [start, *start.parents]
     if (p / "benchmarks" / "compare_intervals.py").exists()), start)
DATA = Path(os.environ.get("FASTCDS_DATA", Path.home() / "Desktop" / "protein2genomic_data"))
V113 = DATA / "validation_v113"     # precomputed REST + TransVar tables (the 5,000-query runs)
V86  = DATA / "validation_v86"
# Matched-pairwise agreement: each tool vs fastCDS on the SAME Ensembl release
# (fastCDS re-indexed to match), same 9-stratum sampler. See benchmarks/agree5k/.
VALM = DATA / "validation_matched"
WORK = DATA / "notebook_software_comparison"; WORK.mkdir(parents=True, exist_ok=True)

def clone(url, dest):
    # Clone a tool's repo once (shallow), then reuse it on later runs.
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(dest)],
                       check=True, capture_output=True, text=True)
    return dest

# Sections 1-2 use the large-N (5,000-query) tables, which take hours to
# regenerate (ensembldb + the rate-limited REST API), so they are read from
# disk.
assert (VALM / "ensembldb_v86.tsv").exists(), "run the matched-pairwise agreement (benchmarks/agree5k/)"
assert (VALM / "transvar_v95.tsv").exists()
assert (VALM / "rest_v115.tsv").exists()
print("repo:", REPO, "| loaded the matched-pairwise agreement tables")

## 1. Per-tool agreement vs fastCDS (Supplementary Table S1)

Matched-pairwise: each comparator runs on its own Ensembl release (fastCDS re-indexed
to match) over the nine-category, 5,000-query validation set. `cds_incomplete` is an
**exclusive** category, so the other eight draw only from complete-CDS transcripts.

- **ensembldb (v86)** and **GenomicFeatures (v86)** - the two Bioconductor
  `proteinToGenome` methods - match fastCDS **5,000 / 5,000 exact in every category**.
- **TransVar (v95)** returns one enclosing genomic span rather than the individual
  CDS intervals, so it is scored at fastCDS's resolution only where every query maps to
  a single CDS block (`single_exon_domain`, `single_exon_gene`); spanning categories
  are **NA**.
- **Ensembl REST (v115)**: 99.06% overall. All 37 coordinate off-by-ones (<= 2 nt) fall
  in `cds_incomplete`; the eight complete-CDS categories have zero off-by-one and zero
  structural mismatches (10 queries were unmapped by the REST service).

In [ ]:
# Supplementary Table S1 - per-tool, per-category agreement with fastCDS
# (finalized 9-category run; built by benchmarks/make_table_s1.py).
table_s1 = pd.read_csv(REPO / "benchmarks" / "table_S1_agreement.tsv", sep="\t")
table_s1

### 1b. Where REST diverges

Because `cds_incomplete` is exclusive, every REST coordinate divergence is confined to
that one category (`cds_start_NF` / `cds_end_NF` transcripts): 163/200 exact, the 37
off-by-ones are all <= 2 nt at the truncated CDS boundary. The eight complete-CDS
categories are clean, and both `proteinToGenome` implementations match fastCDS exactly.

## 2. How fast, and how heavy? Every tool on the same human set

Every tool is run on the **same human v86 query set**, and
line up single-thread throughput and peak memory. fastCDS, ensembldb,
GenomicFeatures and the Ensembl REST API map human protein-domain queries
directly.

geneplot is a general tool (it takes any GFF + domain file), so it goes on the
human set too, with inputs built from the same Ensembl-86 annotation and Pfam
domains (`benchmarks/build_human_tool_inputs.py`, then
`benchmarks/geneplot_human.py`). It has no index, and on the human genome that
bites: geneplot builds a `gffutils` SQLite database of the genome once (~3 min)
then walks each transcript per gene (**~8 genes/s**). On its tiny bundled example
(fruit-fly) it finishes in seconds; on real human data it runs in minutes, which
is why it is measured offline rather than inline. All numbers below are single
thread on the same machine and the same human set.

In [ ]:
# Supplementary Table S2 - mapping speed and peak memory
# (finalized 5-tool run; built by benchmarks/make_table_s2.py).
table_s2 = pd.read_csv(REPO / "benchmarks" / "table_S2_speed_memory.tsv", sep="\t")
table_s2

## Summary

- **fastCDS is orders of magnitude faster than every alternative on human data.**
  End-to-end it maps ~5,900 queries/s (N = 10,000). Of the tools benchmarked for
  speed in **Table S2**, the next-fastest is GenomicFeatures (GRanges) at ~28/s -
  so **fastCDS is more than 200-fold faster than GenomicFeatures** - then geneplot
  ~8/s, ensembldb ~6/s, and the Ensembl REST API ~1.3/s. (TransVar is a span-only
  variant-annotation tool, so it is compared for coordinate accuracy in Table S1,
  not in the speed table.)
- **The bundled-example speed is misleading.** geneplot looks quick on its small
  fruit-fly sample (seconds), but on the human genome it collapses into the
  slow-tool range, because it does not build a reusable index: it rebuilds a
  `gffutils` database and re-reads the domain file per gene. fastCDS pays the genome
  cost once (the index) and then maps in microseconds.
- **And fastCDS agrees with all of them.** On matched annotation (each tool on its
  native Ensembl release, same 9-category 5,000-query sample, Table S1) it is
  **100% exact to both Bioconductor `proteinToGenome` methods - ensembldb (v86) and
  GenomicFeatures (v86) - in every category**, **100% to TransVar (v95) on the
  single-CDS-block categories it can score** (span-only elsewhere), and **99.06% to
  Ensembl REST (v115)** - where every divergence is a `cds_start_NF` incomplete-CDS
  convention, not an error (see the exclusive `cds_incomplete` category, section 1b).
- **Memory:** fastCDS uses ~0.8 GB at N = 10,000 (mostly the loaded index); the
  Bioconductor tools sit at 1.2-1.3 GB (Table S2). Only
  fastCDS returns the per-CDS-exon decomposition; the others give the genomic
  envelope only.
- **Scale:** fastCDS runs to 1,000,000 queries (see [`scaling_and_ram.ipynb`](https://github.com/SotoLF/fastCDS/blob/main/reproduce_paper/notebooks/scaling_and_ram.ipynb)); the Bioconductor tools and geneplot cap at 10,000, and the rate-limited REST API at 1,000.
